[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/32.%20The%20Inter-Terminal%20Transfer%20Optimization%20Problem/P32-Tier-3_executed.ipynb)

---


# 32. The Inter-Terminal Transfer Optimization Problem

## Tier 3: Genetic Algorithm

### Goal and workflow

This notebook uses a genetic algorithm (GA) to search for low-cost assignments of container transfer requests to vehicles. A candidate assignment is represented as a chromosome. The algorithm creates a population of candidates, evaluates their costs, and repeatedly applies tournament selection, single-point crossover, mutation, and elitism. It reports the best candidate found during the run; a genetic algorithm does not certify that this candidate is globally optimal.

The notebook follows this sequence:

1. Define containers, vehicles, distances, and the chromosome representation.
2. Define the cost and capacity-violation penalty used to evaluate a chromosome.
3. Initialize a population and evolve it using the specified genetic operators.
4. Inspect the selected assignment, its cost components, and the search history.

### Model assumptions and scope

Each container is assigned to exactly one vehicle. A vehicle's capacity is treated as a limit on the sum of the weights assigned to it over the entire modeled set of requests. This is an aggregate assignment model: it does not schedule trips, allow repeated trips, or model vehicle availability over time. The example's total container weight is greater than the sum of the vehicle capacities, so no assignment can satisfy every capacity limit under this assumption. The GA therefore searches penalized candidates, and its best candidate must not be described as feasible unless the capacity checks confirm feasibility.

Container ready times contribute a fixed surcharge of 5 cost units per time unit. The example does not include vehicle schedules or service times, so this surcharge is not a scheduling constraint and does not establish whether a transfer can be performed at a particular time. It is constant across assignments for this data set.

The distance matrix gives the distance for each origin-destination pair. Transportation cost is distance multiplied by the assigned vehicle's cost per distance unit. This model does not include empty repositioning, congestion, emissions, or other operational costs.

### Mathematical model used to evaluate candidates

Let \(I\) be the set of containers and \(V\) the set of vehicles. Each chromosome specifies a vehicle assignment \(a(i)\in V\) for every container \(i\in I\). Define the load of vehicle \(v\) by

$$
L_v = \sum_{i\in I: a(i)=v} w_i.
$$

Here, \(w_i\) is the weight of container \(i\), and \(L_v\) is the total assigned weight for vehicle \(v\). The capacity-violation penalty and the candidate's fitness are

$$
P(a) = 100\sum_{v\in V}\max(0, L_v-Q_v),
\qquad
F(a) = \sum_{i\in I} d_i c_{a(i)} + 5\sum_{i\in I} r_i + P(a).
$$

In these expressions, \(Q_v\) is vehicle \(v\)'s capacity, \(d_i\) is the distance for container \(i\)'s route, \(c_v\) is vehicle \(v\)'s cost per distance unit, and \(r_i\) is the container's ready time. The GA minimizes \(F(a)\); lower fitness is better. A chromosome is feasible precisely when \(L_v\le Q_v\) for every vehicle. The penalty encourages the search to reduce capacity excess, but a finite penalty does not make an infeasible assignment feasible.

The code calculates each route distance from the origin and destination in `ITTGAProblem`, accumulates loads in `calculate_fitness`, and stores the objective and feasibility status on the `Chromosome`.

### How the genetic algorithm works

A population is a collection of chromosomes. Tournament selection samples a small group and chooses its lowest-cost member as a parent. Single-point crossover exchanges the assignment suffixes of two parents; mutation may change one container's assigned vehicle. Elitism copies the best chromosomes directly into the next generation, preserving the best candidate found so far.

If there are \(n=|I|\) containers, \(m=|V|\) vehicles, population size \(P\), and \(G\) generations, a chromosome has \(n\) assignment genes and the assignment space has up to \(m^n\) candidates. Fitness evaluation takes \(O(n)\) time. Evaluating offspring across the run takes \(O(GPn)\) time, while the pairwise population-diversity calculation takes \(O(GP^2n)\) time. These bounds describe the implemented operations; they do not guarantee that the GA will find an optimum.

The reported convergence generation is a diagnostic: it marks a window in which the population's best fitness varies by less than the specified tolerance. It is not a proof of optimality or of convergence to a global optimum. No random seed is set, so independent runs may produce different search histories and candidates.

### Interpreting the results

Inspect the best fitness together with its feasibility flag and penalty. A low-cost candidate that violates capacity is not a valid capacity-feasible plan. The fitness plot tracks the best and average population fitness, the diversity plot reports average pairwise Hamming distance between assignment chromosomes, and the load plot compares assigned loads with capacities. The terminal-flow drawing is schematic; its plotted positions are not a scale representation of the distance matrix.

This tier illustrates a population-based metaheuristic with selection, crossover, mutation, and elitism. It is useful for studying how those mechanisms search a discrete assignment space, but it provides neither an optimality certificate nor a feasibility guarantee.

In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
class Container:
    """A container transfer request between two terminals."""

    def __init__(self, id, origin, destination, ready_time, weight):
        self.id = id
        self.origin = origin
        self.destination = destination
        self.ready_time = ready_time
        self.weight = weight

    def __repr__(self):
        return (
            f'Container{self.id}({self.origin}->{self.destination}, '
            f'w:{self.weight}, r:{self.ready_time})'
        )


class Vehicle:
    """A transport resource with an aggregate capacity and unit cost."""

    def __init__(self, id, capacity, cost_per_km):
        self.id = id
        self.capacity = capacity
        self.cost_per_km = cost_per_km

    def __repr__(self):
        return (
            f'Vehicle{self.id}(cap:{self.capacity}, '
            f'cost:{self.cost_per_km})'
        )


class Chromosome:
    """A candidate solution encoded as ordered container-vehicle pairs."""

    def __init__(self, assignments, fitness=None):
        self.assignments = assignments
        self.fitness = fitness
        self.feasible = False
        self.violation_penalty = 0

    def __repr__(self):
        fitness_text = 'unevaluated' if self.fitness is None else f'{self.fitness:.2f}'
        return f'Chromosome(fitness={fitness_text}, feasible={self.feasible})'


class ITTGAProblem:
    """Data and route distances for the inter-terminal assignment problem."""

    def __init__(self, containers, vehicles, distances, terminals):
        self.containers = containers
        self.vehicles = vehicles
        self.distances = distances
        self.terminals = terminals
        self.terminal_to_idx = {
            terminal: index for index, terminal in enumerate(terminals)
        }

        container_ids = [container.id for container in containers]
        vehicle_ids = [vehicle.id for vehicle in vehicles]
        if len(set(container_ids)) != len(container_ids):
            raise ValueError('Container IDs must be unique.')
        if len(set(vehicle_ids)) != len(vehicle_ids):
            raise ValueError('Vehicle IDs must be unique.')
        if not containers or not vehicles:
            raise ValueError('The problem must include containers and vehicles.')
        if distances.shape != (len(terminals), len(terminals)):
            raise ValueError('The distance matrix must match the terminal list.')

        self.container_by_id = {container.id: container for container in containers}
        self.vehicle_by_id = {vehicle.id: vehicle for vehicle in vehicles}
        self.distance_cache = {}

        for container in containers:
            if container.origin not in self.terminal_to_idx:
                raise ValueError(f'Unknown origin terminal: {container.origin}')
            if container.destination not in self.terminal_to_idx:
                raise ValueError(
                    f'Unknown destination terminal: {container.destination}'
                )
            origin_idx = self.terminal_to_idx[container.origin]
            destination_idx = self.terminal_to_idx[container.destination]
            self.distance_cache[container.id] = distances[
                origin_idx, destination_idx
            ]

In [ ]:
class InterTerminalTransferGA:
    """Genetic algorithm for inter-terminal container assignment."""

    def __init__(
        self,
        problem,
        population_size=40,
        generations=150,
        crossover_rate=0.8,
        mutation_rate=0.1,
        elite_size=4,
    ):
        if population_size < 3:
            raise ValueError('Population size must be at least 3.')
        if generations < 0:
            raise ValueError('Generations cannot be negative.')
        if not 0 <= crossover_rate <= 1:
            raise ValueError('Crossover rate must be between 0 and 1.')
        if not 0 <= mutation_rate <= 1:
            raise ValueError('Mutation rate must be between 0 and 1.')
        if not 0 <= elite_size < population_size:
            raise ValueError('Elite size must be nonnegative and below population size.')

        self.problem = problem
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size

        self.population = []
        self.best_chromosome = None
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_generation = None

    def create_random_chromosome(self):
        """Assign each container to a randomly selected vehicle."""
        vehicle_ids = list(self.problem.vehicle_by_id)
        assignments = [
            (container.id, random.choice(vehicle_ids))
            for container in self.problem.containers
        ]
        return Chromosome(assignments)

    def calculate_fitness(self, chromosome):
        """Evaluate transport cost, readiness surcharge, and capacity penalty."""
        expected_container_ids = [
            container.id for container in self.problem.containers
        ]
        assigned_container_ids = [
            container_id for container_id, _ in chromosome.assignments
        ]
        if assigned_container_ids != expected_container_ids:
            raise ValueError(
                'A chromosome must assign each container exactly once '
                'in problem order.'
            )

        vehicle_loads = {
            vehicle_id: 0 for vehicle_id in self.problem.vehicle_by_id
        }
        transport_cost = 0

        for container_id, vehicle_id in chromosome.assignments:
            if vehicle_id not in self.problem.vehicle_by_id:
                raise ValueError(f'Unknown vehicle ID: {vehicle_id}')

            container = self.problem.container_by_id[container_id]
            vehicle = self.problem.vehicle_by_id[vehicle_id]
            distance = self.problem.distance_cache[container_id]
            transport_cost += distance * vehicle.cost_per_km
            vehicle_loads[vehicle_id] += container.weight

        readiness_surcharge = 5 * sum(
            container.ready_time for container in self.problem.containers
        )
        penalty = 100 * sum(
            max(
                0,
                vehicle_loads[vehicle_id]
                - self.problem.vehicle_by_id[vehicle_id].capacity,
            )
            for vehicle_id in vehicle_loads
        )

        chromosome.fitness = transport_cost + readiness_surcharge + penalty
        chromosome.violation_penalty = penalty
        chromosome.feasible = all(
            vehicle_loads[vehicle_id]
            <= self.problem.vehicle_by_id[vehicle_id].capacity
            for vehicle_id in vehicle_loads
        )
        return chromosome.fitness

    def initialize_population(self):
        """Create and rank the initial population of candidate assignments."""
        self.population = []
        for _ in range(self.population_size):
            chromosome = self.create_random_chromosome()
            self.calculate_fitness(chromosome)
            self.population.append(chromosome)

        self.population.sort(key=lambda chromosome: chromosome.fitness)
        self.best_chromosome = copy.deepcopy(self.population[0])

    def tournament_selection(self, tournament_size=3):
        """Choose the lowest-fitness chromosome in a random tournament."""
        tournament = random.sample(
            self.population, min(tournament_size, len(self.population))
        )
        return min(tournament, key=lambda chromosome: chromosome.fitness)

    def crossover(self, parent1, parent2):
        """Apply single-point crossover, or copy parents when it is not used."""
        if (
            random.random() > self.crossover_rate
            or len(parent1.assignments) < 2
        ):
            return copy.deepcopy(parent1), copy.deepcopy(parent2)

        crossover_point = random.randint(1, len(parent1.assignments) - 1)
        offspring1_assignments = (
            parent1.assignments[:crossover_point]
            + parent2.assignments[crossover_point:]
        )
        offspring2_assignments = (
            parent2.assignments[:crossover_point]
            + parent1.assignments[crossover_point:]
        )
        return (
            Chromosome(offspring1_assignments),
            Chromosome(offspring2_assignments),
        )

    def mutate(self, chromosome):
        """Change one assignment with the configured mutation probability."""
        if random.random() > self.mutation_rate:
            return chromosome

        vehicle_ids = list(self.problem.vehicle_by_id)
        if len(vehicle_ids) < 2:
            return chromosome

        mutation_point = random.randrange(len(chromosome.assignments))
        container_id, old_vehicle_id = chromosome.assignments[mutation_point]
        alternatives = [
            vehicle_id
            for vehicle_id in vehicle_ids
            if vehicle_id != old_vehicle_id
        ]
        mutated_assignments = chromosome.assignments.copy()
        mutated_assignments[mutation_point] = (
            container_id,
            random.choice(alternatives),
        )
        return Chromosome(mutated_assignments)

    def calculate_population_diversity(self):
        """Return the average pairwise Hamming distance between chromosomes."""
        if len(self.population) <= 1:
            return 0.0

        total_distance = 0
        comparisons = 0
        for first_index in range(len(self.population)):
            for second_index in range(first_index + 1, len(self.population)):
                distance = sum(
                    first_gene != second_gene
                    for first_gene, second_gene in zip(
                        self.population[first_index].assignments,
                        self.population[second_index].assignments,
                    )
                )
                total_distance += distance
                comparisons += 1

        return total_distance / comparisons

    def evolve_generation(self):
        """Create the next population using elitism and genetic operators."""
        new_population = copy.deepcopy(self.population[:self.elite_size])

        while len(new_population) < self.population_size:
            parent1 = self.tournament_selection()
            parent2 = self.tournament_selection()
            offspring1, offspring2 = self.crossover(parent1, parent2)
            offspring1 = self.mutate(offspring1)
            offspring2 = self.mutate(offspring2)

            self.calculate_fitness(offspring1)
            self.calculate_fitness(offspring2)
            new_population.extend([offspring1, offspring2])

        self.population = new_population[:self.population_size]
        self.population.sort(key=lambda chromosome: chromosome.fitness)

        if self.population[0].fitness < self.best_chromosome.fitness:
            self.best_chromosome = copy.deepcopy(self.population[0])

    def solve(self):
        """Run the GA and return the best candidate and search statistics."""
        print('=== GENETIC ALGORITHM FOR INTER-TERMINAL TRANSFER ===')
        print(f'Population size: {self.population_size}')
        print(f'Generations: {self.generations}')
        print(f'Crossover rate: {self.crossover_rate}')
        print(f'Mutation rate: {self.mutation_rate}')
        print(f'Elite size: {self.elite_size}')

        self.fitness_history = []
        self.diversity_history = []
        self.convergence_generation = None
        start_time = time.time()

        self.initialize_population()
        print(f'\nInitial best fitness: {self.best_chromosome.fitness:.2f}')

        for generation in range(self.generations):
            self.evolve_generation()
            best_fitness = self.population[0].fitness
            average_fitness = sum(
                chromosome.fitness for chromosome in self.population
            ) / len(self.population)
            diversity = self.calculate_population_diversity()

            self.fitness_history.append(
                {
                    'generation': generation,
                    'best_fitness': best_fitness,
                    'avg_fitness': average_fitness,
                    'diversity': diversity,
                }
            )

            if generation > 20 and self.convergence_generation is None:
                recent_best = [
                    record['best_fitness']
                    for record in self.fitness_history[-20:]
                ]
                if max(recent_best) - min(recent_best) < 0.01:
                    self.convergence_generation = generation

            if generation % 10 == 0 or generation == self.generations - 1:
                print(
                    f'Generation {generation:3d}: '
                    f'Best = {best_fitness:8.2f}, '
                    f'Average = {average_fitness:8.2f}, '
                    f'Diversity = {diversity:5.2f}'
                )

        computation_time = time.time() - start_time
        print(f'\nBest fitness found: {self.best_chromosome.fitness:.2f}')
        print(f'Best candidate feasible: {self.best_chromosome.feasible}')
        print(f'Computation time: {computation_time:.3f} seconds')
        if self.convergence_generation is not None:
            print(f'Diagnostic convergence generation: {self.convergence_generation}')

        return self.create_solution_summary(computation_time)

    def create_solution_summary(self, computation_time):
        """Collect assignments, loads, cost components, and search history."""
        vehicle_assignments = {
            vehicle_id: [] for vehicle_id in self.problem.vehicle_by_id
        }
        vehicle_loads = {
            vehicle_id: 0 for vehicle_id in self.problem.vehicle_by_id
        }
        transport_cost = 0

        for container_id, vehicle_id in self.best_chromosome.assignments:
            container = self.problem.container_by_id[container_id]
            vehicle = self.problem.vehicle_by_id[vehicle_id]
            vehicle_assignments[vehicle_id].append(
                {
                    'container_id': container_id,
                    'origin': container.origin,
                    'destination': container.destination,
                    'weight': container.weight,
                    'ready_time': container.ready_time,
                }
            )
            vehicle_loads[vehicle_id] += container.weight
            transport_cost += (
                self.problem.distance_cache[container_id]
                * vehicle.cost_per_km
            )

        readiness_surcharge = 5 * sum(
            container.ready_time for container in self.problem.containers
        )

        return {
            'best_fitness': self.best_chromosome.fitness,
            'feasible': self.best_chromosome.feasible,
            'violation_penalty': self.best_chromosome.violation_penalty,
            'transport_cost': transport_cost,
            'readiness_surcharge': readiness_surcharge,
            'computation_time': computation_time,
            'convergence_generation': self.convergence_generation,
            'vehicle_assignments': vehicle_assignments,
            'vehicle_loads': vehicle_loads,
            'fitness_history': self.fitness_history,
            'final_population_diversity': self.calculate_population_diversity(),
        }

In [ ]:
def create_concrete_example():
    """Build the five-container, two-vehicle example instance."""
    containers = [
        Container(1, 'A', 'B', 0, 2),
        Container(2, 'A', 'C', 0, 1),
        Container(3, 'B', 'C', 1, 2),
        Container(4, 'C', 'A', 0, 1),
        Container(5, 'B', 'A', 2, 2),
    ]

    vehicles = [
        Vehicle(1, 3, 10),
        Vehicle(2, 2, 12),
    ]

    terminals = ['A', 'B', 'C']
    distance_matrix = np.array(
        [
            [0, 3, 5],
            [3, 0, 4],
            [5, 4, 0],
        ]
    )

    return ITTGAProblem(containers, vehicles, distance_matrix, terminals)


problem = create_concrete_example()

total_container_weight = sum(container.weight for container in problem.containers)
total_vehicle_capacity = sum(vehicle.capacity for vehicle in problem.vehicles)
print(f'Total container weight: {total_container_weight}')
print(f'Total vehicle capacity: {total_vehicle_capacity}')
if total_container_weight > total_vehicle_capacity:
    print(
        'The aggregate capacity limits cannot all be satisfied for this '
        'instance; the GA will report a penalized candidate.'
    )

ga = InterTerminalTransferGA(
    problem,
    population_size=40,
    generations=150,
)
solution = ga.solve()

print('\n=== GENETIC ALGORITHM SOLUTION ANALYSIS ===')
print(f"Best candidate fitness: {solution['best_fitness']:.2f}")
print(f"Capacity-feasible: {solution['feasible']}")
print(f"Capacity-violation penalty: {solution['violation_penalty']:.2f}")

In [ ]:
def display_detailed_results(solution, problem):
    """Display assignments, capacity use, objective components, and diagnostics."""
    print('\n=== DETAILED SOLUTION RESULTS ===')

    print('\nVehicle assignments and aggregate loads:')
    for vehicle_id, assignments in solution['vehicle_assignments'].items():
        vehicle = problem.vehicle_by_id[vehicle_id]
        load = solution['vehicle_loads'][vehicle_id]
        utilization = load / vehicle.capacity * 100 if vehicle.capacity else 0

        print(f'\nVehicle {vehicle_id}:')
        print(
            f'  Load: {load}/{vehicle.capacity} '
            f'({utilization:.1f}% of capacity)'
        )
        for assignment in assignments:
            print(
                f"  Container {assignment['container_id']}: "
                f"{assignment['origin']}->{assignment['destination']} "
                f"(weight: {assignment['weight']}, "
                f"ready time: {assignment['ready_time']})"
            )

    recomputed_transport_cost = 0
    for vehicle_id, assignments in solution['vehicle_assignments'].items():
        vehicle = problem.vehicle_by_id[vehicle_id]
        vehicle_cost = sum(
            problem.distance_cache[assignment['container_id']]
            * vehicle.cost_per_km
            for assignment in assignments
        )
        recomputed_transport_cost += vehicle_cost
        print(f'  Vehicle {vehicle_id} transport cost: {vehicle_cost:.2f}')

    assert np.isclose(
        recomputed_transport_cost, solution['transport_cost']
    ), 'The reported transport cost must match the assignments.'

    expected_penalty = 100 * sum(
        max(
            0,
            solution['vehicle_loads'][vehicle_id]
            - problem.vehicle_by_id[vehicle_id].capacity,
        )
        for vehicle_id in solution['vehicle_loads']
    )
    assert np.isclose(
        expected_penalty, solution['violation_penalty']
    ), 'The reported penalty must match the capacity excess.'

    component_total = (
        solution['transport_cost']
        + solution['readiness_surcharge']
        + solution['violation_penalty']
    )
    assert np.isclose(
        component_total, solution['best_fitness']
    ), 'Fitness must equal the sum of its reported components.'

    print('\nObjective components:')
    print(f"  Transport cost: {solution['transport_cost']:.2f}")
    print(f"  Readiness surcharge: {solution['readiness_surcharge']:.2f}")
    print(f"  Capacity-violation penalty: {solution['violation_penalty']:.2f}")
    print(f"  Total fitness: {solution['best_fitness']:.2f}")
    print(f"  Capacity-feasible: {solution['feasible']}")

    print('\nSearch diagnostics:')
    print(f"  Computation time: {solution['computation_time']:.3f} seconds")
    print(
        '  Diagnostic convergence generation: '
        f"{solution['convergence_generation']}"
    )
    print(
        '  Final population diversity: '
        f"{solution['final_population_diversity']:.2f}"
    )
    if not solution['feasible']:
        print(
            '  Interpretation: this candidate violates at least one '
            'aggregate vehicle capacity.'
        )


display_detailed_results(solution, problem)

In [ ]:
def visualize_ga_solution(solution, problem):
    """Plot search progress, assignment loads, and a schematic terminal flow."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Genetic Algorithm Solution Analysis', fontsize=16, fontweight='bold')

    generations = [record['generation'] for record in solution['fitness_history']]
    best_fitness = [record['best_fitness'] for record in solution['fitness_history']]
    average_fitness = [record['avg_fitness'] for record in solution['fitness_history']]
    diversity = [record['diversity'] for record in solution['fitness_history']]

    fitness_axis = axes[0, 0]
    fitness_axis.plot(
        generations, best_fitness, color='blue', linewidth=2, label='Best fitness'
    )
    fitness_axis.plot(
        generations,
        average_fitness,
        color='red',
        linestyle='--',
        linewidth=2,
        label='Average fitness',
    )
    fitness_axis.set_xlabel('Generation')
    fitness_axis.set_ylabel('Fitness (cost units)')
    fitness_axis.set_title('Fitness over generations')
    fitness_axis.legend()
    fitness_axis.grid(True, alpha=0.3)
    if solution['convergence_generation'] is not None:
        fitness_axis.axvline(
            x=solution['convergence_generation'],
            color='green',
            linestyle=':',
            linewidth=2,
            label='Convergence diagnostic',
        )
        fitness_axis.legend()

    diversity_axis = axes[0, 1]
    diversity_axis.plot(generations, diversity, color='green', linewidth=2)
    diversity_axis.set_xlabel('Generation')
    diversity_axis.set_ylabel('Average pairwise Hamming distance')
    diversity_axis.set_title('Population diversity')
    diversity_axis.grid(True, alpha=0.3)

    load_axis = axes[1, 0]
    vehicle_ids = list(solution['vehicle_loads'])
    loads = [solution['vehicle_loads'][vehicle_id] for vehicle_id in vehicle_ids]
    capacities = [
        problem.vehicle_by_id[vehicle_id].capacity for vehicle_id in vehicle_ids
    ]
    positions = np.arange(len(vehicle_ids))
    width = 0.35
    load_axis.bar(
        positions - width / 2,
        loads,
        width,
        label='Assigned load',
        color='skyblue',
    )
    load_axis.bar(
        positions + width / 2,
        capacities,
        width,
        label='Capacity',
        color='lightcoral',
    )
    load_axis.set_xlabel('Vehicle ID')
    load_axis.set_ylabel('Weight units')
    load_axis.set_title('Assigned load and capacity')
    load_axis.set_xticks(positions)
    load_axis.set_xticklabels(vehicle_ids)
    load_axis.legend()
    load_axis.grid(True, alpha=0.3)
    for position, load, capacity in zip(positions, loads, capacities):
        utilization = load / capacity * 100 if capacity else 0
        load_axis.text(
            position,
            max(load, capacity) + 0.2,
            f'{utilization:.0f}%',
            ha='center',
            va='bottom',
            fontweight='bold',
        )

    flow_axis = axes[1, 1]
    terminal_positions = {'A': (0, 0), 'B': (3, 0), 'C': (1.5, 2.5)}
    for terminal, position in terminal_positions.items():
        flow_axis.scatter(
            position[0],
            position[1],
            s=500,
            color='lightblue',
            edgecolors='navy',
            linewidth=2,
        )
        flow_axis.text(
            position[0],
            position[1],
            terminal,
            ha='center',
            va='center',
            fontsize=14,
            fontweight='bold',
        )

    vehicle_colors = plt.get_cmap('tab10')
    vehicle_color_map = {
        vehicle_id: vehicle_colors(index % 10)
        for index, vehicle_id in enumerate(vehicle_ids)
    }
    for vehicle_id, assignments in solution['vehicle_assignments'].items():
        for assignment in assignments:
            origin_position = terminal_positions[assignment['origin']]
            destination_position = terminal_positions[assignment['destination']]
            flow_axis.annotate(
                '',
                xy=destination_position,
                xytext=origin_position,
                arrowprops={
                    'arrowstyle': '->',
                    'color': vehicle_color_map[vehicle_id],
                    'linewidth': 2,
                    'alpha': 0.7,
                },
            )
            midpoint = (
                (origin_position[0] + destination_position[0]) / 2,
                (origin_position[1] + destination_position[1]) / 2,
            )
            flow_axis.text(
                midpoint[0],
                midpoint[1],
                str(assignment['container_id']),
                ha='center',
                va='center',
                fontsize=9,
                bbox={
                    'boxstyle': 'round,pad=0.2',
                    'facecolor': 'white',
                    'alpha': 0.8,
                },
            )

    flow_axis.set_title('Schematic container flows by vehicle')
    flow_axis.set_xlabel('Schematic x-position (not to scale)')
    flow_axis.set_ylabel('Schematic y-position (not to scale)')
    flow_axis.set_aspect('equal')
    flow_axis.grid(True, alpha=0.3)
    legend_elements = [
        plt.Line2D(
            [0],
            [0],
            color=vehicle_color_map[vehicle_id],
            linewidth=2,
            label=f'Vehicle {vehicle_id}',
        )
        for vehicle_id in vehicle_ids
    ]
    flow_axis.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()
    plt.show()
    return fig


fig = visualize_ga_solution(solution, problem)

## Summary and key insights

### Reading the GA output

The returned fitness is the sum of transport cost, the fixed readiness surcharge, and the capacity-violation penalty. The `feasible` flag tests the aggregate capacity limits directly. If it is false, the reported assignment is a penalized search candidate rather than a capacity-feasible operating plan. In this example, the total assigned weight exceeds total available aggregate capacity, so a feasible assignment is impossible under the stated model assumption.

The best-fitness curve describes the best member of each generation's population; elitism also preserves the best candidate found over the run. The average-fitness curve describes the population as a whole. Their movement can reveal search progress, but neither curve establishes global optimality. Average pairwise Hamming distance measures how many assignment genes differ between pairs of chromosomes; a declining value indicates reduced population variation, not necessarily a poor or good solution by itself.

### Algorithm components

| Component | Role in this implementation |
|---|---|
| Chromosome | An ordered list assigning each container to one vehicle |
| Fitness | Transport cost plus readiness surcharge and capacity penalty |
| Tournament selection | Chooses parents by comparing randomly sampled candidates |
| Single-point crossover | Combines assignment segments from two parents |
| Mutation | May change one container's assigned vehicle |
| Elitism | Copies the best current candidates into the next generation |
| Diversity statistic | Tracks average pairwise Hamming distance in the population |

### Scope of conclusions

This run demonstrates the mechanics of a genetic algorithm on a small assignment instance. Its result depends on stochastic choices and the selected GA parameters; the notebook does not set a random seed or provide an optimality certificate. Runtime and fitness from one run are observations of that execution, not general performance guarantees. Comparisons with other solution tiers require their corresponding models, assumptions, and measured runs to be available on the same basis.